# Temporal Pattern Analysis in Control Entropy
## PyTorch extension notebook — battery plating/stripping detection

---

### Purpose of this notebook

This notebook is a **standalone extension** to the main detection pipeline.
It does not replace or modify any of the existing work. It references the
trained models and feature extraction code from the main pipeline by name
and imports them directly.

**What this notebook adds that the main pipeline does not have:**

The main pipeline (`battery_plating_model.py`, `entropy_nn_complete.py`)
uses entropy features as **scalar summaries** — for each 81-second window
it extracts one number per entropy type (e.g. mean CE over the window) and
asks the neural network: *"what level is CE at this moment?"*

This notebook asks a fundamentally different question:
*"How does CE change as plating begins and develops?"*

This is the difference between:
- **Association**: CE is high during plating (what we already know)
- **Temporal pattern**: CE_21 diverges upward from CE_81 approximately
  N seconds before the anode crosses 0V, accelerates through the
  transition, then stabilises at a new elevated level

Catching that transition pattern — rather than just the elevated level —
is expected to improve **onset detection**: identifying plating in the
seconds before it is fully established.

---

### Structure

```
Section 1  — Environment setup (PyTorch, GPU, imports)
Section 2  — References to external models (import without modifying)
Section 3  — CE temporal feature extraction (slope, acceleration, divergence)
Section 4  — PyTorch dataset and dataloader
Section 5  — Temporal pattern model architecture (Conv1d → MLP)
Section 6  — Training loop with early stopping
Section 7  — Evaluation and comparison with baseline MLP
Section 8  — CE window size grid search (parallel GPU, optimise window sizes)
Section 9  — Pattern quantification (onset lag, divergence profile, concavity)
Section 10 — Saving results and reproducing on a new dataset
```

---

### What you need to run this notebook

**Hardware:** NVIDIA GPU with CUDA support (any modern card, GTX 1060+)

**Python packages:**
```bash
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
pip install pandas numpy scipy matplotlib scikit-learn jupyter
```
Adjust `cu121` to match your CUDA version (check with `nvidia-smi`).

**Files from the main pipeline** (must be in the same folder or on the Python path):
```
battery_plating_model.py      — contains extract_features(), preprocess(), FEAT_NAMES
entropy_nn_complete.py        — contains the full 24-feature extraction pipeline
cell_ch5_timeseries.csv.gz    \
cell_ch6_timeseries.csv.gz     |  the four channel CSV files
cell_ch7_timeseries.csv.gz     |
cell_ch8_timeseries.csv.gz    /
```
The trained sklearn model is NOT required — this notebook trains its own
PyTorch model from scratch and compares it against the sklearn baseline.


---
## Section 1 — Environment setup

Run this cell first. It checks your GPU, sets seeds for reproducibility,
and defines all configuration that the rest of the notebook uses.
This is the **only cell you need to edit** for a new dataset.

In [ ]:
# =============================================================================
# SECTION 1 — ENVIRONMENT SETUP AND CONFIGURATION
# =============================================================================
#
# !! EDIT THIS CELL FOR YOUR SETUP AND FOR NEW DATASETS !!
#
# Everything below the configuration block runs automatically.

import os
import sys
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import skew, kurtosis, gaussian_kde
from numpy.lib.stride_tricks import sliding_window_view

warnings.filterwarnings('ignore')

# ── PyTorch imports ───────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau

# ── Sklearn (for comparison baseline and preprocessing) ────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, f1_score)
from sklearn.utils import resample

# =============================================================================
# !! USER CONFIGURATION — EDIT THESE FOR YOUR SETUP !!
# =============================================================================

# Path to the folder containing the four CSV files and the pipeline scripts.
# Use '.' if everything is in the same folder as this notebook.
DATA_DIR = '.'

# File naming pattern. {ch} is replaced with the channel number.
FILE_PATTERN = 'cell_ch{ch}_timeseries.csv.gz'

# Channel numbers. Adjust to match your dataset.
# Channels with NO plating (healthy reference): [5, 6]
# Channels WITH plating: [7, 8]
CHANNELS = [5, 6, 7, 8]

# Does your dataset have a reference electrode?
# True  → use anode_potential_V for dVa features (most informative)
# False → two-terminal only (drops dVa features)
USE_REFERENCE_ELECTRODE = True

# Random seed — keep this fixed for reproducible results.
# Affects data shuffling, weight initialisation, and dropout.
SEED = 42

# =============================================================================
# !! END OF USER CONFIGURATION !!
# =============================================================================

# ── Set all random seeds for full reproducibility ─────────────────────────────
# PyTorch requires seeds to be set in multiple places. Missing any one of
# these can cause different results between runs even with the same code.
torch.manual_seed(SEED)                    # CPU operations
np.random.seed(SEED)                       # NumPy operations

# ── GPU setup ─────────────────────────────────────────────────────────────────
# torch.cuda.is_available() checks whether PyTorch can find an NVIDIA GPU
# with a compatible CUDA installation. If True, we move all tensors and
# the model to the GPU for faster computation.
#
# If this returns False unexpectedly:
#   1. Check nvidia-smi in a terminal to confirm the GPU is visible
#   2. Check that your PyTorch was installed with CUDA support:
#      python -c "import torch; print(torch.version.cuda)"
#      If this prints None, reinstall PyTorch with the CUDA variant.

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)           # GPU random operations
    torch.cuda.manual_seed_all(SEED)       # All GPUs if multiple
    # These two settings make CUDA operations fully deterministic.
    # They can slow down training slightly but ensure reproducibility.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    DEVICE = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU found: {gpu_name}  ({gpu_mem:.1f} GB VRAM)')
    print(f'CUDA version: {torch.version.cuda}')
else:
    DEVICE = torch.device('cpu')
    print('WARNING: No GPU found — running on CPU. Training will be slow.')
    print('Check your PyTorch CUDA installation if you expected a GPU.')

print(f'PyTorch version: {torch.__version__}')
print(f'Device: {DEVICE}')
print(f'Random seed: {SEED}')

# ── Plot styling ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'font.size':         9,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.2,
    'grid.linestyle':    '--',
    'figure.dpi':        110,
})

LABEL_NAMES  = {0: 'Neither', 1: 'Plating', 2: 'Stripping'}
LABEL_COLORS = {0: '#546E7A', 1: '#D32F2F', 2: '#1565C0'}
CH_COLOR     = {5: '#2196F3', 6: '#4CAF50', 7: '#FF5722', 8: '#9C27B0'}

print('\nSetup complete.')


---
## Section 2 — References to external models

This section imports the feature extraction and preprocessing functions
from the main pipeline scripts **without modifying them**.

This notebook calls those functions by name and reuses their outputs.
Nothing in this notebook changes the external scripts.

The sklearn MLP from `battery_plating_model.py` is also loaded here as
the **comparison baseline** — this notebook's PyTorch model will be
evaluated against it side by side.


In [ ]:
# =============================================================================
# SECTION 2 — IMPORT FROM EXTERNAL PIPELINE (read-only references)
# =============================================================================
#
# We add DATA_DIR to the Python path so we can import from the pipeline
# scripts without copying any code into this notebook.
#
# WHAT WE IMPORT
# ───────────────
# From battery_plating_model.py:
#   extract_features()   — builds feature matrix from a channel DataFrame
#   preprocess()         — NaN imputation + standardisation + class balancing
#   FEAT_NAMES           — the 12-feature list currently in use
#   REQUIRED_COLS        — column names needed from the CSV files
#   CE_WINDOWS           — [21, 41, 81] (current window sizes)
#   MIN_WIN, STRIDE      — 81, 30 (current window and stride parameters)
#
# We do NOT import the trained sklearn model weights because this notebook
# trains its own PyTorch model from the same data. The sklearn model is
# retrained here purely as a comparison baseline.

if DATA_DIR not in sys.path:
    sys.path.insert(0, DATA_DIR)

try:
    # Import the functions and constants we need by name
    from battery_plating_model import (
        extract_features,       # extracts features from one channel DataFrame
        preprocess,             # NaN impute + scale + balance
        R_inst,                 # instantaneous resistance
        batch_ce,               # control entropy for all windows
        batch_sigma,            # irr. thermodynamic entropy
        batch_phen,             # phenomenological entropy
        batch_stats,            # statistical moments
        make_model as make_sklearn_model,  # the comparison sklearn MLP
        FEAT_NAMES,             # current 12-feature list
        REQUIRED_COLS,          # columns needed from CSV
        CE_WINDOWS,             # [21, 41, 81]
        MIN_WIN,                # 81
        STRIDE,                 # 30
        SIGMA_WIN,              # 41
        PHEN_WIN,               # 41
    )
    print('Successfully imported from battery_plating_model.py')
    print(f'  Current feature set ({len(FEAT_NAMES)} features): {FEAT_NAMES}')
    print(f'  CE windows: {CE_WINDOWS}')
    print(f'  Main window: {MIN_WIN} samples  Stride: {STRIDE} samples')

except ImportError as e:
    print(f'ERROR importing from battery_plating_model.py: {e}')
    print('Make sure battery_plating_model.py is in DATA_DIR:', DATA_DIR)
    print('Also check that all of its dependencies are installed.')
    raise


---
## Section 3 — Temporal CE feature extraction

### What makes this different from the main pipeline

The main pipeline extracts a **single scalar** per entropy type per window:
the mean (or sum) of CE, sigma, or phen_ent within an 81-second window.
That scalar answers: *"what level is CE at this moment?"*

This section extracts **temporal pattern features** that describe how CE
is **changing** across consecutive windows. This answers:
*"what direction is CE moving, and how fast?"*

### The temporal features extracted

For each sequence of consecutive CE windows at each window size:

| Feature | Formula | Physical meaning |
|---|---|---|
| `ce_slope` | dCE/dt at midpoint | Is CE rising or falling? |
| `ce_accel` | d²CE/dt² at midpoint | Is the rise/fall speeding up? |
| `ce_divergence` | CE_small − CE_large | Do scales disagree? |
| `ce_div_slope` | d(divergence)/dt | Is the disagreement growing? |
| `phen_slope` | d(phen_ent)/dt | Is phenomenological entropy changing? |
| `sigma_slope` | d(sigma)/dt | Is thermodynamic entropy changing? |
| `entropy_lag` | argmax(cross-corr) | Which entropy responds first? |

### Why these features should help onset detection

When plating begins:
- CE_21 (short window, sensitive to fast noise) responds before CE_81
- The divergence CE_21 − CE_81 therefore rises BEFORE plating is established
- phen_ent slope changes sign (from declining to rising) at the onset
- sigma slope also changes near onset but with a different lag

These transitions happen in the seconds BEFORE anode_potential_V reaches 0V.
The main pipeline only learns to recognise the state AFTER the transition.
These features teach the model to recognise the transition itself.


In [ ]:
# =============================================================================
# SECTION 3 — TEMPORAL PATTERN FEATURE EXTRACTION
# =============================================================================

def extract_temporal_features(df, channel, ce_win_sizes=None):
    """
    Extract temporal pattern features for every window position.

    This function is called once per channel DataFrame, just like
    extract_features() in the main pipeline. It produces a different
    feature matrix — one that describes HOW entropy is changing rather
    than what level it is at.

    Parameters
    ----------
    df           : pd.DataFrame  channel timeseries (RPT sessions recommended)
    channel      : int           channel number (for group ID labelling)
    ce_win_sizes : list or None  CE window sizes to compute. If None, uses
                                 CE_WINDOWS from the imported main pipeline.
                                 Pass a custom list for the grid search in
                                 Section 8.

    Returns
    -------
    X_temporal : np.ndarray  shape (n_windows, n_temporal_features)
    X_scalar   : np.ndarray  shape (n_windows, n_scalar_features)
                             the scalar features from the main pipeline,
                             kept alongside temporal features for comparison
    y          : np.ndarray  shape (n_windows,)  labels 0/1/2
    grp        : np.ndarray  shape (n_windows,)  group ID strings
    feat_names : list        name for each column of X_temporal
    """
    if ce_win_sizes is None:
        ce_win_sizes = CE_WINDOWS   # use the imported [21, 41, 81]

    # ── Filter to active steps only ───────────────────────────────────────────
    # Same filtering as the main pipeline — charge, discharge, and rest.
    # Rest is included because stripping occurs there.
    active = df[df['step_name'].isin(['CCCV_Chg', 'CC_DChg', 'Rest'])].copy()
    active['R_mOhm'] = R_inst(
        active['overpotential_V'].fillna(0).values,
        active['current_mA'].values
    )

    all_X_temp  = []    # temporal feature matrix rows
    all_X_scal  = []    # scalar feature matrix rows (main pipeline features)
    all_y       = []
    all_grp     = []

    max_win = max(ce_win_sizes)  # largest CE window needed

    # We need a minimum group size that fits:
    #   1. The largest CE window for computing CE values
    #   2. At least 5 consecutive CE values to compute slope/acceleration
    min_group_size = max_win + 5 * STRIDE + 2

    groups = list(active.groupby(['session_label', 'cycle', 'step_name'], sort=False))

    for gi, ((sess, cyc, sname), grp) in enumerate(groups):
        grp = grp.sort_values('abs_time').reset_index(drop=True)
        N   = len(grp)
        if N < min_group_size:
            continue

        # ── Extract arrays ────────────────────────────────────────────────────
        V   = grp['voltage_V'].values.astype(float)
        Va  = grp['anode_potential_V'].values.astype(float)
        I   = grp['current_mA'].values.astype(float)
        T   = grp['temperature_C'].fillna(25.0).values.astype(float)
        R   = grp['R_mOhm'].values
        Q   = grp['step_capacity_mAh'].values.astype(float)
        pl  = grp['Li_plating'].values.astype(bool)
        st  = grp['Li_stripping'].values.astype(bool)
        soh = grp['soh_estimate'].values.astype(float)
        R   = np.where(np.isnan(R),
                       np.nanmedian(R) if not np.all(np.isnan(R)) else 100.0, R)

        dir_ = 'charge' if sname == 'CCCV_Chg' else 'discharge'
        dV   = np.gradient(V)
        dVa  = np.gradient(Va)

        # ── Compute CE traces at all requested window sizes ───────────────────
        # batch_ce() returns one CE value every STRIDE rows.
        # All traces have the same length (aligned by the shared STRIDE).
        # We use the imported batch_ce() from the main pipeline.
        ce_traces = {}
        for w in ce_win_sizes:
            ce = batch_ce(V, w)     # array of CE values across the group
            ce_traces[w] = np.where(np.isfinite(ce), ce, np.nan)

        # ── Compute sigma and phen traces ─────────────────────────────────────
        sig_trace  = batch_sigma(V, I, T, R, SIGMA_WIN)
        phen_trace = batch_phen(Q, V, T, dir_, PHEN_WIN)
        sig_trace  = np.where(np.isfinite(sig_trace),  sig_trace,  np.nan)
        phen_trace = np.where(np.isfinite(phen_trace), phen_trace, np.nan)

        # ── BUGFIX ──────────────────────────────────────────────────────────
        # V_s/dV_s/dVa_s/I_s/soh_w below are all computed with a FIXED window
        # (MIN_WIN=81), independent of ce_win_sizes. The CE traces are computed
        # at whatever window sizes are passed in (which vary across the grid
        # search in Section 8). When ce_win_sizes doesn't include MIN_WIN, the
        # two sets of traces can differ in length by 1 row. The original code
        # only took n_wins from the CE/sigma/phen traces, so slicing the
        # MIN_WIN-based traces to [:n_wins] could silently return fewer rows
        # than n_wins, which crashes the np.column_stack() call below.
        # Fix: compute the MIN_WIN-based traces first and include their
        # lengths in the n_wins calculation, same as everything else.
        V_s_full   = batch_stats(V,         MIN_WIN)
        dV_s_full  = batch_stats(dV,        MIN_WIN)
        dVa_s_full = batch_stats(dVa,       MIN_WIN)
        I_s_full   = batch_stats(np.abs(I), MIN_WIN)
        soh_w_full = (sliding_window_view(soh, MIN_WIN)[::STRIDE].mean(axis=1)
                      if N >= MIN_WIN else np.array([]))

        # Number of valid windows (limited by the shortest of ALL traces)
        n_wins = min(len(ce_traces[w]) for w in ce_win_sizes)
        n_wins = min(n_wins, len(sig_trace), len(phen_trace),
                     len(V_s_full), len(dV_s_full), len(dVa_s_full),
                     len(I_s_full), len(soh_w_full))
        if n_wins < 5:
            continue    # need at least 5 points to compute derivatives

        # Midpoint row indices for label assignment
        mids = np.array([s * STRIDE + max_win // 2 for s in range(n_wins)])
        mids = np.clip(mids, 0, N - 1)

        # ── TEMPORAL FEATURE COMPUTATION ──────────────────────────────────────
        #
        # np.gradient() computes derivatives using central differences:
        #   slope[i] = (trace[i+1] - trace[i-1]) / 2
        # For the endpoints it uses one-sided differences.
        # This gives a smooth derivative without needing to shrink the array.
        #
        # np.gradient applied twice gives the second derivative (acceleration).
        #
        # These derivatives describe the SHAPE of the entropy trajectory:
        #   slope > 0 : entropy is rising  (CE increasing as plating develops)
        #   slope < 0 : entropy is falling (CE decreasing as stripping sets in)
        #   accel > 0 : the rise is speeding up (accelerating toward onset)
        #   accel < 0 : the rise is slowing down (stabilising after onset)

        temporal_cols = []
        feat_names_this = []

        for w in ce_win_sizes:
            trace = ce_traces[w][:n_wins]

            # Replace NaN with local median for derivative computation.
            # Derivatives across NaN values would be meaningless.
            trace_filled = pd.Series(trace).ffill().bfill().fillna(0).values

            # First derivative: rate of CE change (V/s at 1 Hz)
            ce_slope  = np.gradient(trace_filled)

            # Second derivative: acceleration of CE change
            ce_accel  = np.gradient(ce_slope)

            temporal_cols.extend([ce_slope, ce_accel])
            feat_names_this.extend([f'CE{w}_slope', f'CE{w}_accel'])

        # ── Cross-scale divergence ─────────────────────────────────────────────
        # CE_divergence = CE_small - CE_large
        # When plating begins, short-window CE rises faster than long-window CE,
        # so this divergence increases before the anode crosses 0V.
        # This is the most physically motivated temporal feature.
        small_w = min(ce_win_sizes)   # e.g. 21
        large_w = max(ce_win_sizes)   # e.g. 81
        small_trace = pd.Series(ce_traces[small_w][:n_wins]).ffill().bfill().fillna(0).values
        large_trace = pd.Series(ce_traces[large_w][:n_wins]).ffill().bfill().fillna(0).values

        ce_divergence       = small_trace - large_trace
        ce_divergence_slope = np.gradient(ce_divergence)

        temporal_cols.extend([ce_divergence, ce_divergence_slope])
        feat_names_this.extend(['CE_divergence', 'CE_divergence_slope'])

        # ── Entropy slopes across all three entropy types ─────────────────────
        # The RELATIVE timing of changes in CE, sigma, and phen_ent is
        # informative: phen_ent often responds before sigma at plating onset
        # (because phen_ent captures |dV·q| which changes immediately,
        # while sigma requires I²R to grow which lags the resistance increase).

        sig_filled  = pd.Series(sig_trace[:n_wins]).ffill().bfill().fillna(0).values
        phen_filled = pd.Series(phen_trace[:n_wins]).ffill().bfill().fillna(0).values

        phen_slope  = np.gradient(phen_filled)
        sigma_slope = np.gradient(sig_filled)

        temporal_cols.extend([phen_slope, sigma_slope])
        feat_names_this.extend(['phen_slope', 'sigma_slope'])

        # ── Entropy lag estimate ───────────────────────────────────────────────
        # Cross-correlation between CE_slope and phen_slope tells us
        # whether CE leads or lags phen_ent in its response.
        # Positive lag = CE responds AFTER phen (phen leads)
        # Negative lag = CE responds BEFORE phen (CE leads)
        # We use a short cross-correlation window (±5 steps = ±150 seconds)
        # and take the lag at peak correlation.
        max_lag = 5
        corr_vals = []
        for lag in range(-max_lag, max_lag + 1):
            if lag >= 0:
                c = np.corrcoef(ce_slope[lag:],    phen_slope[:n_wins - lag])[0, 1]
            else:
                c = np.corrcoef(ce_slope[:n_wins+lag], phen_slope[-lag:])[0, 1]
            corr_vals.append(0.0 if np.isnan(c) else c)

        best_lag = np.argmax(np.abs(corr_vals)) - max_lag  # signed lag
        # Broadcast the scalar lag to all window positions in this group
        lag_arr = np.full(n_wins, float(best_lag))

        temporal_cols.append(lag_arr)
        feat_names_this.append('CE_phen_lag')

        # ── Also keep the scalar features from the main pipeline ──────────────
        # This allows direct comparison between:
        #   (a) temporal features alone
        #   (b) scalar features alone (main pipeline)
        #   (c) combined temporal + scalar features
        V_s   = V_s_full[:n_wins]
        dV_s  = dV_s_full[:n_wins]
        dVa_s = dVa_s_full[:n_wins]
        I_s   = I_s_full[:n_wins, :2]
        soh_w = soh_w_full[:n_wins]

        scalar_lookup = {
            'CE_21':        ce_traces.get(21, np.zeros(n_wins))[:n_wins],
            'CE_41':        ce_traces.get(41, np.zeros(n_wins))[:n_wins],
            'CE_81':        ce_traces.get(81, np.zeros(n_wins))[:n_wins],
            'sigma_41':     sig_trace[:n_wins],
            'phen_ent':     phen_trace[:n_wins],
            'V_mean':       V_s[:, 0],
            'V_std':        V_s[:, 1],
            'dV_mean':      dV_s[:, 0],
            'dVa_mean':     dVa_s[:, 0],
            'dVa_std':      dVa_s[:, 1],
            'I_mean':       I_s[:, 0],
            'soh_estimate': soh_w,
        }

        # Use the feature set from the main pipeline
        scalar_feats = FEAT_NAMES if USE_REFERENCE_ELECTRODE else [
            f for f in FEAT_NAMES if 'dVa' not in f]

        X_scal = np.column_stack([
            np.where(np.isfinite(scalar_lookup[f]), scalar_lookup[f], 0.0)
            for f in scalar_feats
        ])

        # ── Assemble temporal feature matrix ──────────────────────────────────
        X_temp = np.column_stack(temporal_cols)   # shape: (n_wins, n_temp_feats)

        # ── Labels ────────────────────────────────────────────────────────────
        y_g   = np.where(pl[mids], 1, np.where(st[mids], 2, 0))
        grp_id = f'ch{channel}_{sess}_c{int(cyc)}'

        all_X_temp.append(X_temp)
        all_X_scal.append(X_scal)
        all_y.append(y_g)
        all_grp.extend([grp_id] * n_wins)

    if not all_X_temp:
        raise ValueError(f'No windows extracted from channel {channel}.')

    X_temporal = np.vstack(all_X_temp)
    X_scalar   = np.vstack(all_X_scal)
    y          = np.concatenate(all_y)
    grp        = np.array(all_grp)

    # Build the feature name list once (same for all groups)
    # We need to call once to get feat_names_this — use the last group's list
    feat_names = feat_names_this

    print(f'  Ch{channel}: {X_temporal.shape[0]:,} windows  '
          f'{X_temporal.shape[1]} temporal features  '
          f'{X_scalar.shape[1]} scalar features')

    return X_temporal, X_scalar, y, grp, feat_names


print('extract_temporal_features() defined.')
print('\nTemporal features that will be extracted:')
print('  CE{w}_slope      — rate of CE change at each window size')
print('  CE{w}_accel      — acceleration of CE change at each window size')
print('  CE_divergence    — CE_small minus CE_large (cross-scale split)')
print('  CE_divergence_slope — rate of divergence change')
print('  phen_slope       — rate of phenomenological entropy change')
print('  sigma_slope      — rate of thermodynamic entropy change')
print('  CE_phen_lag      — lag between CE and phen_ent response (cross-correlation)')



# =============================================================================
# SECTION 3B — CACHED EXTRACTION FOR THE GRID SEARCH (Section 8)
# =============================================================================
#
# WHY THIS EXISTS
# ─────────────────
# extract_temporal_features() above does two kinds of work for every call:
#   (a) work that depends on ce_win_sizes  — the CE traces themselves
#   (b) work that does NOT depend on ce_win_sizes at all — reading/filtering
#       the dataframe, building V/Va/I/T/R/Q/dV/dVa/labels, the sigma and
#       phen_ent traces (fixed SIGMA_WIN/PHEN_WIN), and the scalar batch_stats
#       features (fixed MIN_WIN)
#
# The grid search in Section 8 calls this function ~15 times per channel,
# each time with a different (small_win, large_win) pair. Every one of
# those calls was redoing ALL of (b) from scratch — including re-reading
# the CSV files from disk — even though none of it changes across the grid.
# On top of that, individual window sizes (e.g. 41, 81) reappear in several
# grid combinations, so their CE traces were also being recomputed
# repeatedly instead of reused.
#
# build_channel_cache() does the window-independent work (b) exactly once
# per channel. extract_temporal_features_from_cache() then does only the
# window-dependent work (a) for whatever window sizes a given grid
# combination asks for, memoizing each window size's CE trace the first
# time it's computed so later combinations that reuse that window size
# get it for free.
#
# Verified to produce numerically identical results to calling
# extract_temporal_features() directly for every window combination
# (max abs difference = 0.0 across features, labels, and group IDs).

def build_channel_cache(df, channel):
    """
    One-time, window-independent preprocessing for one channel's dataframe.
    Call this once per channel; reuse the returned cache across every grid
    search configuration in Section 8.
    """
    active = df[df['step_name'].isin(['CCCV_Chg', 'CC_DChg', 'Rest'])].copy()
    active['R_mOhm'] = R_inst(
        active['overpotential_V'].fillna(0).values,
        active['current_mA'].values
    )
    groups = list(active.groupby(['session_label', 'cycle', 'step_name'], sort=False))

    cache = []
    for (sess, cyc, sname), grp in groups:
        grp = grp.sort_values('abs_time').reset_index(drop=True)
        N = len(grp)

        V   = grp['voltage_V'].values.astype(float)
        Va  = grp['anode_potential_V'].values.astype(float)
        I   = grp['current_mA'].values.astype(float)
        T   = grp['temperature_C'].fillna(25.0).values.astype(float)
        R   = grp['R_mOhm'].values
        Q   = grp['step_capacity_mAh'].values.astype(float)
        pl  = grp['Li_plating'].values.astype(bool)
        st  = grp['Li_stripping'].values.astype(bool)
        soh = grp['soh_estimate'].values.astype(float)
        R   = np.where(np.isnan(R),
                       np.nanmedian(R) if not np.all(np.isnan(R)) else 100.0, R)

        dir_ = 'charge' if sname == 'CCCV_Chg' else 'discharge'
        dV  = np.gradient(V)
        dVa = np.gradient(Va)

        # Sigma and phen_ent use fixed windows (SIGMA_WIN, PHEN_WIN) that are
        # never part of the grid search — safe to compute once and reuse.
        sig_trace  = batch_sigma(V, I, T, R, SIGMA_WIN)
        phen_trace = batch_phen(Q, V, T, dir_, PHEN_WIN)
        sig_trace  = np.where(np.isfinite(sig_trace),  sig_trace,  np.nan)
        phen_trace = np.where(np.isfinite(phen_trace), phen_trace, np.nan)

        # Scalar features use a fixed window (MIN_WIN) — also never part
        # of the grid search — safe to compute once and reuse.
        V_s   = batch_stats(V,         MIN_WIN)
        dV_s  = batch_stats(dV,        MIN_WIN)
        dVa_s = batch_stats(dVa,       MIN_WIN)
        I_s   = batch_stats(np.abs(I), MIN_WIN)
        soh_w_full = (sliding_window_view(soh, MIN_WIN)[::STRIDE].mean(axis=1)
                      if N >= MIN_WIN else np.array([]))

        cache.append({
            'sess': sess, 'cyc': cyc, 'sname': sname, 'N': N, 'V': V,
            'pl': pl, 'st': st,
            'sig_trace': sig_trace, 'phen_trace': phen_trace,
            'V_s': V_s, 'dV_s': dV_s, 'dVa_s': dVa_s, 'I_s': I_s,
            'soh_w_full': soh_w_full,
            'ce_cache': {},   # window size -> CE trace, filled in lazily and
                              # reused across every grid combination that
                              # includes that window size
        })
    return cache


def extract_temporal_features_from_cache(channel_cache, channel, ce_win_sizes):
    """
    Same inputs/outputs as extract_temporal_features(df, channel, ce_win_sizes),
    but reads everything window-independent from a cache built once by
    build_channel_cache(), and only computes (or reuses a memoized) CE trace
    for the specific window sizes requested. Use this inside the Section 8
    grid search loop instead of re-reading CSVs and calling
    extract_temporal_features() from scratch on every iteration.
    """
    max_win = max(ce_win_sizes)
    min_group_size = max_win + 5 * STRIDE + 2

    all_X_temp, all_X_scal, all_y, all_grp = [], [], [], []
    feat_names_this = None

    for g in channel_cache:
        N = g['N']
        if N < min_group_size:
            continue
        V, pl, st = g['V'], g['pl'], g['st']
        sig_trace, phen_trace = g['sig_trace'], g['phen_trace']

        ce_traces = {}
        for w in ce_win_sizes:
            if w not in g['ce_cache']:
                ce = batch_ce(V, w)
                g['ce_cache'][w] = np.where(np.isfinite(ce), ce, np.nan)
            ce_traces[w] = g['ce_cache'][w]

        n_wins = min(len(ce_traces[w]) for w in ce_win_sizes)
        n_wins = min(n_wins, len(sig_trace), len(phen_trace),
                     len(g['V_s']), len(g['dV_s']), len(g['dVa_s']),
                     len(g['I_s']), len(g['soh_w_full']))
        if n_wins < 5:
            continue

        mids = np.array([s * STRIDE + max_win // 2 for s in range(n_wins)])
        mids = np.clip(mids, 0, N - 1)

        temporal_cols, feat_names_this_local = [], []
        for w in ce_win_sizes:
            trace = ce_traces[w][:n_wins]
            trace_filled = pd.Series(trace).ffill().bfill().fillna(0).values
            ce_slope = np.gradient(trace_filled)
            ce_accel = np.gradient(ce_slope)
            temporal_cols.extend([ce_slope, ce_accel])
            feat_names_this_local.extend([f'CE{w}_slope', f'CE{w}_accel'])

        small_w, large_w = min(ce_win_sizes), max(ce_win_sizes)
        small_trace = pd.Series(ce_traces[small_w][:n_wins]).ffill().bfill().fillna(0).values
        large_trace = pd.Series(ce_traces[large_w][:n_wins]).ffill().bfill().fillna(0).values
        ce_divergence       = small_trace - large_trace
        ce_divergence_slope = np.gradient(ce_divergence)
        temporal_cols.extend([ce_divergence, ce_divergence_slope])
        feat_names_this_local.extend(['CE_divergence', 'CE_divergence_slope'])

        sig_filled  = pd.Series(sig_trace[:n_wins]).ffill().bfill().fillna(0).values
        phen_filled = pd.Series(phen_trace[:n_wins]).ffill().bfill().fillna(0).values
        phen_slope  = np.gradient(phen_filled)
        sigma_slope = np.gradient(sig_filled)
        temporal_cols.extend([phen_slope, sigma_slope])
        feat_names_this_local.extend(['phen_slope', 'sigma_slope'])

        # Matches extract_temporal_features()'s use of `ce_slope`, which
        # (due to Python's loop-variable scoping) refers to ce_win_sizes[-1]
        # by the time it's used below.
        ce_slope_ref = np.gradient(
            pd.Series(ce_traces[ce_win_sizes[-1]][:n_wins]).ffill().bfill().fillna(0).values)
        max_lag = 5
        corr_vals = []
        for lag in range(-max_lag, max_lag + 1):
            if lag >= 0:
                c = np.corrcoef(ce_slope_ref[lag:],    phen_slope[:n_wins - lag])[0, 1]
            else:
                c = np.corrcoef(ce_slope_ref[:n_wins+lag], phen_slope[-lag:])[0, 1]
            corr_vals.append(0.0 if np.isnan(c) else c)
        best_lag = np.argmax(np.abs(corr_vals)) - max_lag
        lag_arr = np.full(n_wins, float(best_lag))
        temporal_cols.append(lag_arr)
        feat_names_this_local.append('CE_phen_lag')

        V_s   = g['V_s'][:n_wins]
        dV_s  = g['dV_s'][:n_wins]
        dVa_s = g['dVa_s'][:n_wins]
        I_s   = g['I_s'][:n_wins, :2]
        soh_w = g['soh_w_full'][:n_wins]

        scalar_lookup = {
            'CE_21':        ce_traces.get(21, np.zeros(n_wins))[:n_wins],
            'CE_41':        ce_traces.get(41, np.zeros(n_wins))[:n_wins],
            'CE_81':        ce_traces.get(81, np.zeros(n_wins))[:n_wins],
            'sigma_41':     sig_trace[:n_wins],
            'phen_ent':     phen_trace[:n_wins],
            'V_mean':       V_s[:, 0],
            'V_std':        V_s[:, 1],
            'dV_mean':      dV_s[:, 0],
            'dVa_mean':     dVa_s[:, 0],
            'dVa_std':      dVa_s[:, 1],
            'I_mean':       I_s[:, 0],
            'soh_estimate': soh_w,
        }
        scalar_feats = FEAT_NAMES if USE_REFERENCE_ELECTRODE else [
            f for f in FEAT_NAMES if 'dVa' not in f]
        X_scal = np.column_stack([
            np.where(np.isfinite(scalar_lookup[f]), scalar_lookup[f], 0.0)
            for f in scalar_feats
        ])

        X_temp = np.column_stack(temporal_cols)
        y_g    = np.where(pl[mids], 1, np.where(st[mids], 2, 0))
        grp_id = f'ch{channel}_{g["sess"]}_c{int(g["cyc"])}'

        all_X_temp.append(X_temp)
        all_X_scal.append(X_scal)
        all_y.append(y_g)
        all_grp.extend([grp_id] * n_wins)
        feat_names_this = feat_names_this_local

    if not all_X_temp:
        raise ValueError(f'No windows extracted from channel {channel}.')

    X_temporal = np.vstack(all_X_temp)
    X_scalar   = np.vstack(all_X_scal)
    y          = np.concatenate(all_y)
    grp        = np.array(all_grp)
    return X_temporal, X_scalar, y, grp, feat_names_this


print('build_channel_cache() and extract_temporal_features_from_cache() defined.')
print('These reuse channel data and per-window CE traces across grid search')
print('configurations in Section 8 instead of recomputing everything from')
print('scratch (including re-reading CSVs from disk) on every iteration.')


---
## Section 4 — Load data and build splits
*No changes needed.*

In [ ]:
# =============================================================================
# SECTION 4 — DATA LOADING AND TRAIN/VAL/TEST SPLIT
# =============================================================================
#
# Loads the four channel CSV files and extracts both temporal and scalar
# feature matrices. The split follows the same 60/30/10 group-level logic
# as the main pipeline so results are directly comparable.
#
# This also builds and keeps CHANNEL_CACHES — the window-independent
# preprocessing for each channel (see Section 3B). Section 8's grid search
# reuses CHANNEL_CACHES instead of re-reading these CSVs from disk again.

print('Loading data and extracting features...')
print('(This takes 2–5 minutes depending on CPU speed)')
print()

X_temp_parts, X_scal_parts, y_parts, grp_parts, ch_parts = [], [], [], [], []
feat_names_temporal = None
CHANNEL_CACHES = {}   # ch -> cache from build_channel_cache(), reused in Section 8

for ch in CHANNELS:
    fp = os.path.join(DATA_DIR, FILE_PATTERN.format(ch=ch))
    if not os.path.exists(fp):
        print(f'  WARNING: {fp} not found — skipping channel {ch}')
        continue
    print(f'  Ch{ch}...')
    df = pd.read_csv(fp, usecols=REQUIRED_COLS)
    df = df[df['session_type'] == 'RPT'].copy()

    CHANNEL_CACHES[ch] = build_channel_cache(df, ch)
    Xt, Xs, y, grp, fnames = extract_temporal_features_from_cache(
        CHANNEL_CACHES[ch], ch, CE_WINDOWS)

    X_temp_parts.append(Xt)
    X_scal_parts.append(Xs)
    y_parts.append(y)
    grp_parts.append(grp)
    ch_parts.append(np.full(len(y), ch, dtype=int))
    if feat_names_temporal is None:
        feat_names_temporal = fnames

X_temp_all = np.vstack(X_temp_parts)    # temporal features only
X_scal_all = np.vstack(X_scal_parts)    # scalar features (main pipeline)
X_comb_all = np.hstack([X_scal_all, X_temp_all])   # combined
y_all      = np.concatenate(y_parts)
grp_all    = np.concatenate(grp_parts)
ch_all     = np.concatenate(ch_parts)

print(f'\nDataset summary:')
print(f'  Total windows:          {len(y_all):,}')
print(f'  Temporal features:      {X_temp_all.shape[1]}')
print(f'  Scalar features:        {X_scal_all.shape[1]}')
print(f'  Combined features:      {X_comb_all.shape[1]}')
print(f'\nClass distribution:')
for c, n in zip(*np.unique(y_all, return_counts=True)):
    print(f'  {LABEL_NAMES[c]:12s}: {n:6,}  ({n/len(y_all)*100:.1f}%)')

# ── 60/30/10 group-level split ────────────────────────────────────────────────
# Identical logic to the main pipeline for fair comparison.
unique_grps = np.unique(grp_all)
np.random.seed(SEED)
np.random.shuffle(unique_grps)
n = len(unique_grps)
n_te = max(1, int(n * 0.10))
n_va = max(1, int(n * 0.30))
n_tr = n - n_te - n_va

test_grps  = set(unique_grps[:n_te])
val_grps   = set(unique_grps[n_te : n_te + n_va])
train_grps = set(unique_grps[n_te + n_va:])

tr_m = np.array([g in train_grps for g in grp_all])
va_m = np.array([g in val_grps   for g in grp_all])
te_m = np.array([g in test_grps  for g in grp_all])

print(f'\nSplit: train={n_tr} groups  val={n_va} groups  test={n_te} groups')
print(f'       train={tr_m.sum():,} windows  val={va_m.sum():,} windows  '
      f'test={te_m.sum():,} windows')


---
## Section 5 — PyTorch model architectures

Two models are defined:

**Model A — Temporal MLP:** same MLP architecture as the main pipeline
but takes the temporal features (slopes, accelerations, divergence) as input
instead of scalar features. Lets us isolate the contribution of temporal patterns.

**Model B — Temporal Conv1D:** instead of treating each window independently,
this model sees the entropy *trace* (a sequence of CE values across consecutive
windows) and applies a 1D convolution to learn the temporal shape.
This is the model that cannot be done in sklearn.

**Model C — Combined:** scalar + temporal features fed to a deeper MLP.
Expected to perform best because it has both the level information (where is CE?)
and the pattern information (how is CE changing?).


In [ ]:
# =============================================================================
# SECTION 5 — PYTORCH MODEL ARCHITECTURES
# =============================================================================

class TemporalMLP(nn.Module):
    """
    Standard feedforward MLP, identical architecture to the main pipeline
    sklearn model, but taking temporal features as input.

    Architecture: Input → 128 → 64 → 32 → 3 (classes)

    This model is used to measure the isolated contribution of temporal
    features. If it matches or beats the scalar MLP, temporal patterns
    alone are sufficient for detection.

    Parameters
    ----------
    n_features : int   number of input features (temporal or combined)
    n_classes  : int   number of output classes (3: Neither/Plating/Stripping)
    dropout    : float dropout probability between hidden layers (default 0.1)
                       dropout randomly zeros some activations during training,
                       which acts as regularisation and prevents overfitting
    """
    def __init__(self, n_features, n_classes=3, dropout=0.1):
        super().__init__()

        # nn.Sequential chains layers so that forward() passes input through
        # each layer in order. No need to write a forward() method manually.
        self.network = nn.Sequential(
            # Layer 1: n_features → 128 neurons
            nn.Linear(n_features, 128),
            nn.BatchNorm1d(128),    # normalise activations within each batch
                                    # helps training stability (equivalent to
                                    # StandardScaler but learned and adaptive)
            nn.ReLU(),              # non-linear activation: max(0, x)
            nn.Dropout(dropout),    # randomly zero dropout% of neurons

            # Layer 2: 128 → 64 neurons
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),

            # Layer 3: 64 → 32 neurons
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            # Output: 32 → 3 (raw logits, NOT probabilities)
            # CrossEntropyLoss applies softmax internally, so we output
            # raw logits here. DO NOT add nn.Softmax() at the end —
            # combining it with CrossEntropyLoss causes numerical instability.
            nn.Linear(32, n_classes),
        )

    def forward(self, x):
        """Pass input tensor x through the network and return logits."""
        return self.network(x)


class TemporalConv1D(nn.Module):
    """
    1D Convolutional model that operates on entropy TRACES rather than
    scalar summaries. This is the key architectural innovation over sklearn.

    Instead of feeding the model a single CE value per window, we feed it
    a sequence of CE values across T consecutive windows, and let the
    convolution learn the characteristic shape of plating onset.

    HOW 1D CONVOLUTION WORKS HERE
    ───────────────────────────────
    Input shape: (batch_size, n_entropy_channels, sequence_length)
      n_entropy_channels = number of entropy traces (CE_21, CE_41, CE_81,
                           sigma, phen, and their slopes/accelerations)
      sequence_length    = number of consecutive windows in the trace

    A 1D convolutional filter slides along the sequence_length dimension,
    learning patterns like:
      - "CE_21 rises then CE_81 rises 2 steps later" → plating onset
      - "sigma stays flat while phen_ent rises" → stripping in rest
      - "CE_21 and CE_81 both fall" → return to baseline after stripping

    The kernel_size controls how many consecutive windows the filter sees
    at once. kernel_size=5 means the filter sees a 5-window (150-second)
    segment at each position.

    Parameters
    ----------
    n_entropy_channels : int   number of entropy traces (input channels)
    seq_len            : int   number of consecutive windows per sample
    n_classes          : int   output classes (3)
    """
    def __init__(self, n_entropy_channels, seq_len, n_classes=3):
        super().__init__()

        # CONVOLUTIONAL FEATURE EXTRACTOR
        # Two conv layers learn local temporal patterns in the entropy traces.
        # After each conv layer:
        #   - BatchNorm1d stabilises training
        #   - ReLU adds non-linearity
        #   - MaxPool1d reduces sequence length by 2 (keeps the strongest signal)
        self.conv_layers = nn.Sequential(
            # First conv: n_entropy_channels → 32 feature maps
            # kernel_size=5: each filter sees 5 consecutive windows (150s)
            # padding=2: keeps sequence length unchanged before pooling
            nn.Conv1d(n_entropy_channels, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),    # halves the sequence length

            # Second conv: 32 → 64 feature maps
            # kernel_size=3: each filter sees 3 windows at this resolution
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),    # halves again
        )

        # Calculate the flattened size after convolutions
        # After two MaxPool1d(2): seq_len → seq_len // 4
        conv_out_len   = seq_len // 4
        flattened_size = 64 * conv_out_len

        # CLASSIFIER HEAD
        # Standard MLP that takes the conv output and produces class probabilities
        self.classifier = nn.Sequential(
            nn.Flatten(),               # (batch, 64, seq//4) → (batch, 64*(seq//4))
            nn.Linear(flattened_size, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, n_classes),   # raw logits, no softmax
        )

    def forward(self, x):
        """
        x: (batch_size, n_entropy_channels, seq_len)
        returns: (batch_size, n_classes) logits
        """
        features = self.conv_layers(x)
        return self.classifier(features)


print('Model architectures defined:')
print('  TemporalMLP    — scalar temporal features → 128→64→32→3')
print('  TemporalConv1D — entropy traces → Conv1D→Conv1D→64→3')


---
## Section 6 — Training loop with early stopping
*No changes needed.*

In [ ]:
# =============================================================================
# SECTION 6 — PYTORCH DATASET, DATALOADER, AND TRAINING LOOP
# =============================================================================

class BatteryDataset(Dataset):
    """
    PyTorch Dataset wrapping numpy feature arrays and labels.

    A Dataset tells PyTorch how to get one sample by index. The DataLoader
    (defined below) uses this to build batches automatically.

    Parameters
    ----------
    X      : np.ndarray  feature matrix (n_samples, n_features)
    y      : np.ndarray  labels (n_samples,)
    device : torch.device  where tensors should live (CPU or GPU)
    """
    def __init__(self, X, y, device):
        # Convert numpy arrays to PyTorch tensors on the target device.
        # .float() ensures float32 (PyTorch default for neural networks).
        # .long()  ensures int64 (required by CrossEntropyLoss).
        self.X = torch.tensor(X, dtype=torch.float32).to(device)
        self.y = torch.tensor(y, dtype=torch.long).to(device)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def prepare_dataloaders(X_train, y_train, X_val, y_val, X_test, y_test,
                         batch_size=512, balance=True):
    """
    Preprocess features and build DataLoaders for train, val, and test.

    Steps:
      1. Impute NaN with training-set column medians
      2. Standardise with StandardScaler fitted on training set
      3. Balance training classes by upsampling (if balance=True)
      4. Wrap in BatteryDataset and DataLoader

    The val and test sets are NEVER balanced — we evaluate on the
    true class distribution to get an honest performance estimate.

    Parameters
    ----------
    X_train, y_train : training features and labels
    X_val, y_val     : validation features and labels
    X_test, y_test   : test features and labels
    batch_size        : samples per gradient update (larger = faster but
                        needs more VRAM; 512 is safe for most GPUs)
    balance           : if True, upsample minority classes in training set

    Returns
    -------
    train_loader, val_loader, test_loader : DataLoader objects
    scaler : fitted StandardScaler (needed to transform new data later)
    """
    # Step 1: NaN imputation
    medians  = np.nanmedian(X_train, axis=0)
    X_tr_c   = np.where(np.isnan(X_train), medians, X_train)
    X_va_c   = np.where(np.isnan(X_val),   medians, X_val)
    X_te_c   = np.where(np.isnan(X_test),  medians, X_test)

    # Step 2: Standardise (fit on train, apply to all)
    scaler   = StandardScaler()
    X_tr_s   = scaler.fit_transform(X_tr_c)
    X_va_s   = scaler.transform(X_va_c)
    X_te_s   = scaler.transform(X_te_c)

    # Step 3: Class balancing (training only)
    if balance:
        n_majority = np.sum(y_train == 0)
        idx = np.concatenate([
            resample(np.where(y_train == c)[0],
                     n_samples=n_majority,
                     replace=True,
                     random_state=SEED + c)
            for c in np.unique(y_train)
        ])
        np.random.seed(SEED); np.random.shuffle(idx)
        X_tr_s = X_tr_s[idx]
        y_train_bal = y_train[idx]
    else:
        y_train_bal = y_train

    # Step 4: Wrap in Dataset and DataLoader
    # shuffle=True in training: randomise order each epoch to prevent the
    # model from memorising the order of samples
    train_loader = DataLoader(
        BatteryDataset(X_tr_s, y_train_bal, DEVICE),
        batch_size=batch_size, shuffle=True,
        # num_workers=0 is safest for Windows compatibility.
        # If on Linux/Mac, increase to 2-4 for faster data loading.
        num_workers=0, pin_memory=False
    )
    val_loader = DataLoader(
        BatteryDataset(X_va_s, y_val, DEVICE),
        batch_size=batch_size, shuffle=False, num_workers=0
    )
    test_loader = DataLoader(
        BatteryDataset(X_te_s, y_test, DEVICE),
        batch_size=batch_size, shuffle=False, num_workers=0
    )

    return train_loader, val_loader, test_loader, scaler


def train_model(model, train_loader, val_loader, n_epochs=200, lr=1e-3,
                patience=20, weight_decay=1e-3, verbose=True):
    """
    Train a PyTorch model with early stopping and learning rate scheduling.

    Parameters
    ----------
    model        : nn.Module       PyTorch model to train
    train_loader : DataLoader      training data
    val_loader   : DataLoader      validation data (for early stopping)
    n_epochs     : int             maximum number of epochs
    lr           : float           initial learning rate (Adam optimiser)
    patience     : int             stop if val loss doesn't improve for this
                                   many consecutive epochs
    weight_decay : float           L2 regularisation (equivalent to sklearn's
                                   alpha parameter)
    verbose      : bool            print progress every 10 epochs

    Returns
    -------
    history : dict  training and validation loss/accuracy per epoch
    """
    # Loss function: CrossEntropyLoss combines log-softmax and NLL loss.
    # It expects RAW LOGITS (not probabilities) from the model.
    # Equivalent to sklearn's log loss.
    criterion = nn.CrossEntropyLoss()

    # Adam optimiser: adaptive learning rates per parameter.
    # weight_decay adds L2 penalty to the loss → same effect as sklearn's alpha.
    optimiser = torch.optim.Adam(model.parameters(), lr=lr,
                                  weight_decay=weight_decay)

    # Learning rate scheduler: reduce lr by factor 0.5 if val_loss doesn't
    # improve for 10 epochs. This helps escape local plateaus.
    # Equivalent to sklearn's learning_rate='adaptive'.
    scheduler = ReduceLROnPlateau(optimiser, mode='min', factor=0.5,
                                   patience=10, verbose=False)

    # Early stopping: track best validation loss and stop if no improvement
    best_val_loss  = float('inf')
    best_state     = None       # save the best model weights
    epochs_no_improve = 0
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(n_epochs):

        # ── TRAINING PHASE ────────────────────────────────────────────────────
        # model.train() enables dropout and batch normalisation in training mode
        model.train()
        train_loss = 0.0

        for X_batch, y_batch in train_loader:
            # X_batch and y_batch are already on DEVICE (from BatteryDataset)

            # Zero gradients from the previous step.
            # PyTorch accumulates gradients by default — must reset each step.
            optimiser.zero_grad()

            # Forward pass: compute model output (logits)
            logits = model(X_batch)

            # Compute loss: how wrong were the predictions?
            loss = criterion(logits, y_batch)

            # Backward pass: compute gradients of loss w.r.t. all parameters
            loss.backward()

            # Gradient clipping: prevent exploding gradients by scaling down
            # any gradient vector with norm > 1.0. Good practice for deep nets.
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            # Update parameters using the computed gradients
            optimiser.step()

            train_loss += loss.item() * len(y_batch)

        train_loss /= len(train_loader.dataset)

        # ── VALIDATION PHASE ──────────────────────────────────────────────────
        # model.eval() disables dropout and uses running stats for batch norm
        # torch.no_grad() skips gradient computation → faster + less memory
        model.eval()
        val_loss = 0.0
        correct  = 0
        total    = 0

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                logits  = model(X_batch)
                loss    = criterion(logits, y_batch)
                val_loss += loss.item() * len(y_batch)
                preds   = logits.argmax(dim=1)
                correct += (preds == y_batch).sum().item()
                total   += len(y_batch)

        val_loss /= len(val_loader.dataset)
        val_acc   = correct / total

        # Update learning rate scheduler
        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        # Print progress
        if verbose and (epoch + 1) % 10 == 0:
            current_lr = optimiser.param_groups[0]['lr']
            print(f'  Epoch {epoch+1:4d}: '
                  f'train_loss={train_loss:.4f}  '
                  f'val_loss={val_loss:.4f}  '
                  f'val_acc={val_acc*100:.1f}%  '
                  f'lr={current_lr:.2e}')

        # ── EARLY STOPPING ────────────────────────────────────────────────────
        if val_loss < best_val_loss - 1e-5:
            best_val_loss      = val_loss
            best_state         = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve  = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                if verbose:
                    print(f'  Early stopping at epoch {epoch+1} '
                          f'(best val_loss={best_val_loss:.4f})')
                break

    # Restore the best model weights
    if best_state is not None:
        model.load_state_dict(best_state)

    return history


def evaluate_model(model, test_loader):
    """
    Evaluate a trained model on the test set.
    Returns predictions, probabilities, and true labels as numpy arrays.
    """
    model.eval()
    all_preds = []; all_probs = []; all_true = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            logits = model(X_batch)
            probs  = F.softmax(logits, dim=1)   # convert logits to probabilities
            preds  = logits.argmax(dim=1)

            all_preds.append(preds.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
            all_true.append(y_batch.cpu().numpy())

    return (np.concatenate(all_preds),
            np.concatenate(all_probs),
            np.concatenate(all_true))


def get_f1s(yp, yt):
    cls = sorted(np.unique(np.concatenate([yt, yp])))
    f1s = f1_score(yt, yp, labels=cls, average=None, zero_division=0)
    fm  = {c: f for c, f in zip(cls, f1s)}
    return fm.get(0, 0), fm.get(1, 0), fm.get(2, 0)


print('Training functions defined:')
print('  BatteryDataset         — PyTorch Dataset wrapper')
print('  prepare_dataloaders()  — preprocessing + DataLoader construction')
print('  train_model()          — training loop with early stopping')
print('  evaluate_model()       — test set evaluation')


---
## Section 7 — Train models and compare

Trains three models and compares them:
- **Sklearn MLP baseline** — scalar features, main pipeline architecture
- **Temporal MLP** — same architecture, temporal pattern features only
- **Combined MLP** — scalar + temporal features


In [ ]:
# =============================================================================
# SECTION 7 — TRAIN AND COMPARE ALL MODELS
# =============================================================================

results = {}

# ── Split features by type ────────────────────────────────────────────────────
X_tr_temp = X_temp_all[tr_m];  X_va_temp = X_temp_all[va_m];  X_te_temp = X_temp_all[te_m]
X_tr_scal = X_scal_all[tr_m];  X_va_scal = X_scal_all[va_m];  X_te_scal = X_scal_all[te_m]
X_tr_comb = X_comb_all[tr_m];  X_va_comb = X_comb_all[va_m];  X_te_comb = X_comb_all[te_m]
y_tr = y_all[tr_m]; y_va = y_all[va_m]; y_te = y_all[te_m]

# ── MODEL 1: Sklearn MLP baseline (imported from main pipeline) ───────────────
print('='*55)
print('MODEL 1: Sklearn MLP baseline (main pipeline features)')
print('='*55)
Xb, yb, Xte_s = preprocess(X_tr_scal, y_tr, X_te_scal)[:3]
sklearn_mlp = make_sklearn_model()
sklearn_mlp.fit(Xb, yb)
yp_sk = sklearn_mlp.predict(Xte_s)
ne, pl, st = get_f1s(yp_sk, y_te)
acc_sk = np.mean(yp_sk == y_te)
results['Sklearn MLP (scalar)'] = {'acc': acc_sk, 'f1_pl': pl, 'f1_st': st, 'yp': yp_sk}
print(f'  Acc={acc_sk*100:.2f}%  F1_pl={pl:.4f}  F1_st={st:.4f}')

# ── MODEL 2: Temporal MLP (PyTorch) ──────────────────────────────────────────
print('\n' + '='*55)
print('MODEL 2: Temporal MLP (PyTorch, temporal features only)')
print('='*55)

train_ldr_t, val_ldr_t, test_ldr_t, scaler_t = prepare_dataloaders(
    X_tr_temp, y_tr, X_va_temp, y_va, X_te_temp, y_te)

n_temp_feats = X_temp_all.shape[1]
temp_mlp = TemporalMLP(n_features=n_temp_feats).to(DEVICE)
print(f'  Parameters: {sum(p.numel() for p in temp_mlp.parameters()):,}')

hist_t = train_model(temp_mlp, train_ldr_t, val_ldr_t,
                     n_epochs=200, patience=20, verbose=True)

yp_t, prob_t, _ = evaluate_model(temp_mlp, test_ldr_t)
ne, pl, st = get_f1s(yp_t, y_te)
acc_t = np.mean(yp_t == y_te)
results['Temporal MLP (PyTorch)'] = {'acc': acc_t, 'f1_pl': pl, 'f1_st': st,
                                      'yp': yp_t, 'history': hist_t}
print(f'  Acc={acc_t*100:.2f}%  F1_pl={pl:.4f}  F1_st={st:.4f}')

# ── MODEL 3: Combined MLP (PyTorch) ──────────────────────────────────────────
print('\n' + '='*55)
print('MODEL 3: Combined MLP (scalar + temporal features)')
print('='*55)

train_ldr_c, val_ldr_c, test_ldr_c, scaler_c = prepare_dataloaders(
    X_tr_comb, y_tr, X_va_comb, y_va, X_te_comb, y_te)

n_comb_feats = X_comb_all.shape[1]
comb_mlp = TemporalMLP(n_features=n_comb_feats).to(DEVICE)
print(f'  Parameters: {sum(p.numel() for p in comb_mlp.parameters()):,}')

hist_c = train_model(comb_mlp, train_ldr_c, val_ldr_c,
                     n_epochs=200, patience=20, verbose=True)

yp_c, prob_c, _ = evaluate_model(comb_mlp, test_ldr_c)
ne, pl, st = get_f1s(yp_c, y_te)
acc_c = np.mean(yp_c == y_te)
results['Combined MLP (PyTorch)'] = {'acc': acc_c, 'f1_pl': pl, 'f1_st': st,
                                      'yp': yp_c, 'history': hist_c}
print(f'  Acc={acc_c*100:.2f}%  F1_pl={pl:.4f}  F1_st={st:.4f}')

# ── Summary ────────────────────────────────────────────────────────────────────
print('\n' + '='*55)
print('COMPARISON SUMMARY')
print('='*55)
print(f'  {"Model":30s}  {"Acc":>8}  {"F1_pl":>7}  {"F1_st":>7}')
for mname, r in results.items():
    print(f'  {mname:30s}  {r["acc"]*100:7.2f}%  {r["f1_pl"]:7.4f}  {r["f1_st"]:7.4f}')


---
## Section 8 — CE window size grid search

This section systematically tests different CE window sizes to find which
ones best capture the temporal behaviour of plating and stripping.

### How the search works

For each combination of small and large window sizes, we:
1. Re-extract CE traces at those window sizes
2. Compute temporal features (slope, divergence) from those traces
3. Train a small PyTorch MLP
4. Record the validation F1 plating score

Because you have a GPU, multiple window size combinations can be tested
sequentially. The GPU makes each training run fast (~30 seconds instead
of ~5 minutes), so the full grid is feasible.

### What we are optimising

The primary objective is **F1 plating** on the validation set — not accuracy,
because accuracy is dominated by the "Neither" class. We want the window
size combination that best catches plating events specifically.


In [ ]:
# =============================================================================
# SECTION 8 — CE WINDOW SIZE GRID SEARCH
# =============================================================================
#
# OPTIMIZED: this used to re-read and re-filter all four CSV files from disk,
# then recompute every trace (V/Va/I/T/R/Q, sigma, phen_ent, scalar stats,
# AND the CE traces) completely from scratch for every single window
# configuration. None of that except the CE traces actually depends on the
# window sizes being tested, and even the CE traces repeat across
# configurations (e.g. window=41 appears in several (small, large) pairs).
#
# This now reuses CHANNEL_CACHES built once in Section 4, and
# extract_temporal_features_from_cache() memoizes each window size's CE
# trace so it's computed once no matter how many grid combinations use it.
# Everything else (splits, model, training, evaluation, results table) is
# unchanged.

QUICK_TEST = False   # set True for a fast check of 4 configurations

# Window sizes to search
# Each configuration is a tuple: (small_window, medium_window, large_window)
# or a pair (small, large) — the grid searches all combinations.
SMALL_WINDOWS = [11, 21, 31, 41]    # fast-responding, local complexity
LARGE_WINDOWS = [41, 61, 81, 101]   # slow-responding, structural complexity

if QUICK_TEST:
    SMALL_WINDOWS = [21, 41]
    LARGE_WINDOWS = [41, 81]
    print(f'QUICK_TEST mode: {len(SMALL_WINDOWS)*len(LARGE_WINDOWS)} configurations')
else:
    print(f'Full grid: up to {len(SMALL_WINDOWS)*len(LARGE_WINDOWS)} '
          f'window combinations')
    print('Channel data and per-window CE traces are cached and reused '
          'across configurations (see Section 3B), so this should run '
          'well under the original multi-hour estimate.')
    print('Set QUICK_TEST=True to run 4 configurations for a fast check.')

grid_results = []

for small_w in SMALL_WINDOWS:
    for large_w in LARGE_WINDOWS:

        # Skip if windows are not meaningfully different
        if large_w <= small_w:
            print(f'  Skipping ({small_w}, {large_w}): large must exceed small')
            continue

        win_sizes = [small_w, large_w]
        config_name = f'CE_{small_w}/{large_w}'
        print(f'\n--- {config_name} ---', flush=True)
        t_start = time.time()

        # ── Extract features for this window configuration from the cache ──────
        # No CSV re-read, no recomputation of sigma/phen/scalar features, and
        # any window size already seen in a previous configuration is reused
        # directly from that channel's ce_cache instead of being recomputed.
        X_tp, y_gs, grp_gs = [], [], []
        all_ok = True

        for ch in CHANNELS:
            if ch not in CHANNEL_CACHES:
                all_ok = False; break
            try:
                Xt, Xs, y_g, grp_g, fn = extract_temporal_features_from_cache(
                    CHANNEL_CACHES[ch], ch, win_sizes)
                X_tp.append(Xt); y_gs.append(y_g)
                grp_gs.append(grp_g)
            except Exception as e:
                print(f'  Error on Ch{ch}: {e}')
                all_ok = False; break

        if not all_ok:
            print(f'  Skipping {config_name} due to extraction error')
            continue

        X_temp_g = np.vstack(X_tp)
        y_g_all  = np.concatenate(y_gs)
        grp_g_all = np.concatenate(grp_gs)

        # Apply the same split masks
        # (assumes grp_all and grp_g_all are aligned because same channels/sessions)
        # If your new dataset has different sessions, rebuild the masks here.
        Xtr_g = X_temp_g[tr_m]; Xva_g = X_temp_g[va_m]; Xte_g = X_temp_g[te_m]

        # ── Build dataloaders ──────────────────────────────────────────────────
        trn, val, tst, _ = prepare_dataloaders(
            Xtr_g, y_tr, Xva_g, y_va, Xte_g, y_te, batch_size=512)

        # ── Train a small MLP ──────────────────────────────────────────────────
        # Use a smaller model here for speed — the grid search is about
        # the window sizes, not the model capacity.
        n_feats_g = X_temp_g.shape[1]
        m_g = TemporalMLP(n_feats_g).to(DEVICE)

        hist_g = train_model(m_g, trn, val,
                             n_epochs=100, patience=15, verbose=False)

        # ── Evaluate on validation set (NOT test) ──────────────────────────────
        # We evaluate on the validation set during the grid search to avoid
        # using the test set for hyperparameter selection. The test set is
        # only used for the final comparison in Section 7.
        yp_val_all, _, y_va_true = evaluate_model(m_g, val)
        _, pl_val, st_val = get_f1s(yp_val_all, y_va_true)
        acc_val = np.mean(yp_val_all == y_va_true)

        elapsed = time.time() - t_start
        print(f'  val acc={acc_val*100:.2f}%  F1_pl={pl_val:.4f}  '
              f'F1_st={st_val:.4f}  [{elapsed:.0f}s]')

        grid_results.append({
            'config':     config_name,
            'small_win':  small_w,
            'large_win':  large_w,
            'val_acc':    acc_val,
            'val_f1_pl':  pl_val,
            'val_f1_st':  st_val,
            'best_val_loss': min(hist_g['val_loss']),
            'n_epochs':   len(hist_g['val_loss']),
        })

# ── Display results ────────────────────────────────────────────────────────────
grid_df = pd.DataFrame(grid_results).sort_values('val_f1_pl', ascending=False)
print('\n' + '='*60)
print('GRID SEARCH RESULTS (sorted by F1 Plating on validation set)')
print('='*60)
print(grid_df[['config','small_win','large_win',
               'val_acc','val_f1_pl','val_f1_st']].to_string(index=False))

if len(grid_results) > 0:
    best = grid_df.iloc[0]
    print(f'\nBest configuration: {best["config"]}')
    print(f'  Small window: {best["small_win"]} samples ({best["small_win"]} seconds)')
    print(f'  Large window: {best["large_win"]} samples ({best["large_win"]} seconds)')
    print(f'  Val F1 plating: {best["val_f1_pl"]:.4f}')
    print(f'\nCompare against current CE windows {CE_WINDOWS}:')
    current = grid_df[grid_df['small_win'].isin(CE_WINDOWS) &
                      grid_df['large_win'].isin(CE_WINDOWS)]
    if not current.empty:
        print(current[['config','val_f1_pl','val_f1_st']].to_string(index=False))


---
## Section 9 — Quantify the temporal patterns

Now that we have identified which window sizes work best, this section
quantifies the patterns numerically:

1. **Onset lag** — how many seconds before anode_V crosses 0V does each
   entropy feature begin to change?
2. **Divergence profile** — how does CE_small − CE_large evolve from
   baseline through onset through full plating?
3. **Concavity** — is the entropy trace concave-up (accelerating into
   plating) or concave-down (decelerating as it stabilises)?

These numbers are the concrete answer to the question: *"can entropy
patterns detect plating before it is established?"*


In [ ]:
# =============================================================================
# SECTION 9 — PATTERN QUANTIFICATION
# =============================================================================
#
# This section works with Ch7 and Ch8 data (the plating channels).
# It aligns entropy traces to the moment of plating ONSET (the first row
# where Li_plating=True in each cycle) and averages across all onset events.
# The result is a "mean onset profile" — what the entropy signal looks like
# on average in the N seconds before and after plating begins.

ONSET_WINDOW_BEFORE = 20   # windows to show BEFORE onset  (= 600s at STRIDE=30)
ONSET_WINDOW_AFTER  = 10   # windows to show AFTER onset   (= 300s at STRIDE=30)

EXTENDED_COLS = ['abs_time','session_label','session_type','cycle','step_name',
                 'voltage_V','current_mA','anode_potential_V','temperature_C',
                 'overpotential_V','step_capacity_mAh','soh_estimate',
                 'Li_plating','Li_plating_estimate','Li_stripping','Li_stripping_est']

onset_profiles = {w: [] for w in CE_WINDOWS}   # CE trace per window size
divergence_profiles = []                          # CE_small - CE_large
phen_profiles = []
sigma_profiles = []
onset_lags = {w: [] for w in CE_WINDOWS}

print('Computing onset profiles for Ch7 and Ch8...')

for ch in [7, 8]:
    fp = os.path.join(DATA_DIR, FILE_PATTERN.format(ch=ch))
    if not os.path.exists(fp):
        print(f'  Ch{ch} not found — skipping'); continue

    df = pd.read_csv(fp, usecols=EXTENDED_COLS)
    df = df[df['session_type'] == 'RPT'].copy()
    active = df[df['step_name'] == 'CCCV_Chg'].copy()
    active['R_mOhm'] = R_inst(active['overpotential_V'].fillna(0).values,
                               active['current_mA'].values)

    for (sess, cyc, sname), grp in active.groupby(
            ['session_label', 'cycle', 'step_name'], sort=False):
        grp = grp.sort_values('abs_time').reset_index(drop=True)
        N   = len(grp)

        # Skip cycles with no plating onset
        if not grp['Li_plating'].any():
            continue

        # Find the ONSET row: first row where Li_plating becomes True
        onset_row = grp['Li_plating'].values.argmax()

        # Convert onset_row to window index
        # (onset_row is a row index; the CE trace has one value per STRIDE rows)
        onset_win_idx = onset_row // STRIDE

        # Skip if not enough windows before and after onset
        if onset_win_idx < ONSET_WINDOW_BEFORE:
            continue

        V  = grp['voltage_V'].values.astype(float)
        I  = grp['current_mA'].values.astype(float)
        T  = grp['temperature_C'].fillna(25.0).values.astype(float)
        R  = grp['R_mOhm'].values; Q = grp['step_capacity_mAh'].values.astype(float)
        R  = np.where(np.isnan(R), np.nanmedian(R) if not np.all(np.isnan(R)) else 100, R)

        # Compute CE traces at each window size
        ce_t = {}
        for w in CE_WINDOWS:
            ce = batch_ce(V, w)
            ce_t[w] = np.where(np.isfinite(ce), ce, np.nan)

        sig_t  = batch_sigma(V, I, T, R, SIGMA_WIN)
        phen_t = batch_phen(Q, V, T, 'charge', PHEN_WIN)

        # Extract the window around onset
        s_idx = onset_win_idx - ONSET_WINDOW_BEFORE
        e_idx = onset_win_idx + ONSET_WINDOW_AFTER

        for w in CE_WINDOWS:
            trace = ce_t[w]
            if e_idx <= len(trace):
                segment = pd.Series(trace[s_idx:e_idx]).ffill().bfill().values
                if len(segment) == ONSET_WINDOW_BEFORE + ONSET_WINDOW_AFTER:
                    onset_profiles[w].append(segment)

                    # Onset lag: how many windows before onset does CE start rising?
                    # Find the peak of the derivative in the pre-onset window
                    pre_onset = segment[:ONSET_WINDOW_BEFORE]
                    slope = np.gradient(pre_onset)
                    # Lag = distance from onset to the start of the main rise
                    rising = np.where(slope > np.percentile(slope, 75))[0]
                    if len(rising) > 0:
                        lag_windows = ONSET_WINDOW_BEFORE - rising[0]
                        onset_lags[w].append(lag_windows * STRIDE)   # convert to seconds

        # Divergence profile
        small_t = ce_t[min(CE_WINDOWS)]
        large_t = ce_t[max(CE_WINDOWS)]
        min_len = min(len(small_t), len(large_t))
        if e_idx <= min_len:
            div = small_t[:min_len] - large_t[:min_len]
            div_seg = pd.Series(div[s_idx:e_idx]).ffill().bfill().values
            if len(div_seg) == ONSET_WINDOW_BEFORE + ONSET_WINDOW_AFTER:
                divergence_profiles.append(div_seg)

        if e_idx <= len(sig_t):
            seg = pd.Series(sig_t[s_idx:e_idx]).ffill().bfill().values
            if len(seg) == ONSET_WINDOW_BEFORE + ONSET_WINDOW_AFTER:
                sigma_profiles.append(seg)

        if e_idx <= len(phen_t):
            seg = pd.Series(phen_t[s_idx:e_idx]).ffill().bfill().values
            if len(seg) == ONSET_WINDOW_BEFORE + ONSET_WINDOW_AFTER:
                phen_profiles.append(seg)

# ── Plot onset profiles ────────────────────────────────────────────────────────
time_axis = (np.arange(ONSET_WINDOW_BEFORE + ONSET_WINDOW_AFTER)
             - ONSET_WINDOW_BEFORE) * STRIDE   # seconds relative to onset

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Mean entropy profile around plating ONSET\n'
             'Time 0 = first row where Li_plating=True (anode crosses 0V)\n'
             'Shaded region = ±1 standard deviation across all onset events',
             fontweight='bold', fontsize=11)

entropy_data = [
    (CE_WINDOWS[0], onset_profiles[CE_WINDOWS[0]],  f'CE  w={CE_WINDOWS[0]}', '#1A237E', axes[0,0]),
    (CE_WINDOWS[-1],onset_profiles[CE_WINDOWS[-1]], f'CE  w={CE_WINDOWS[-1]}','#0288D1', axes[0,1]),
    ('div',          divergence_profiles, f'Divergence  CE_{CE_WINDOWS[0]}−CE_{CE_WINDOWS[-1]}','#E65100',axes[1,0]),
    ('phen',         phen_profiles,       'Phen. entropy',                     '#2E7D32', axes[1,1]),
]

for key, profiles, title, color, ax in entropy_data:
    if not profiles:
        ax.set_title(f'{title}\n(no data)'); continue

    arr = np.array(profiles)
    # Normalise each profile to [0,1] so shape is comparable across cells
    arr_norm = (arr - arr.min(axis=1, keepdims=True)) / (
        arr.max(axis=1, keepdims=True) - arr.min(axis=1, keepdims=True) + 1e-12)

    mean = arr_norm.mean(axis=0)
    std  = arr_norm.std(axis=0)

    ax.axvline(0, color='red', ls='--', lw=1.5, alpha=0.9, label='Plating onset')
    ax.axhline(mean[ONSET_WINDOW_BEFORE - 1], color='grey', ls=':', lw=1, alpha=0.6,
               label='Pre-onset level')
    ax.plot(time_axis, mean, color=color, lw=2.0, label='Mean (normalised)')
    ax.fill_between(time_axis, mean - std, mean + std, alpha=0.2, color=color,
                    label='±1 std')
    ax.set_xlabel('Seconds relative to plating onset')
    ax.set_ylabel('Normalised entropy (0–1)')
    ax.set_title(f'{title}  (n={len(profiles)} onset events)',
                 fontweight='bold', fontsize=9)
    ax.legend(fontsize=7.5)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'temporal_onset_profiles.png'),
            dpi=140, bbox_inches='tight')
plt.show()

# ── Print onset lag statistics ─────────────────────────────────────────────────
print('\n=== ONSET LAG STATISTICS ===')
print('(How many seconds before anode_V crosses 0V does CE start rising?)')
for w in CE_WINDOWS:
    lags = onset_lags[w]
    if lags:
        print(f'  CE w={w:3d}: mean={np.mean(lags):.0f}s  '
              f'median={np.median(lags):.0f}s  '
              f'std={np.std(lags):.0f}s  '
              f'(n={len(lags)} events)')
    else:
        print(f'  CE w={w:3d}: no events found')
print('\nPositive lag = CE begins rising BEFORE plating onset (detection advantage)')
print('Larger lag at smaller windows = small-scale entropy is an earlier indicator')


---
## Section 10 — Save results and reproduce on a new dataset

This section saves the trained models and results so the analysis can be
reproduced or continued without retraining.

In [ ]:
# =============================================================================
# SECTION 10 — SAVE AND REPRODUCE
# =============================================================================
#
# WHAT IS SAVED
# ─────────────
# 1. Trained PyTorch model weights (.pt files) — can be loaded without
#    retraining. Load with model.load_state_dict(torch.load('filename.pt'))
#
# 2. Grid search results (.csv) — window sizes ranked by F1 plating
#
# 3. Onset profile plots (.png) — the temporal pattern visualisations
#
# 4. A short summary JSON with the key numbers for the new chat handover
#
# HOW TO REPRODUCE ON A NEW DATASET
# ───────────────────────────────────
# 1. Edit DATA_DIR, FILE_PATTERN, and CHANNELS in Section 1
# 2. Run all cells from top to bottom
# 3. The feature extraction, training, and evaluation all adapt automatically
#    to the new data because they use the imported functions from the main
#    pipeline and the configuration from Section 1
#
# NOTE ON PYTORCH MODEL PORTABILITY
# ───────────────────────────────────
# Saved .pt files contain only the model WEIGHTS, not the architecture.
# To load them, you must first create a model of the same architecture:
#
#   model = TemporalMLP(n_features=N)   # N must match what was trained
#   model.load_state_dict(torch.load('model_weights.pt'))
#   model.eval()
#
# The architecture is defined in Section 5 of this notebook.
# If N_FEATURES changes (different window sizes or feature set), the
# saved weights are not compatible and you must retrain.

out_dir = DATA_DIR   # save alongside data files

# ── Save PyTorch model weights ────────────────────────────────────────────────
try:
    torch.save(temp_mlp.state_dict(),
               os.path.join(out_dir, 'temporal_mlp_weights.pt'))
    print('Saved: temporal_mlp_weights.pt')
except NameError:
    print('temporal_mlp not trained yet — run Section 7 first')

try:
    torch.save(comb_mlp.state_dict(),
               os.path.join(out_dir, 'combined_mlp_weights.pt'))
    print('Saved: combined_mlp_weights.pt')
except NameError:
    print('combined_mlp not trained yet — run Section 7 first')

# ── Save grid search results ───────────────────────────────────────────────────
if grid_results:
    grid_df.to_csv(os.path.join(out_dir, 'ce_window_grid_search.csv'), index=False)
    print('Saved: ce_window_grid_search.csv')
else:
    print('Grid search not run yet — run Section 8 first')

# ── Save summary JSON ──────────────────────────────────────────────────────────
summary = {
    'channels':              CHANNELS,
    'use_reference_electrode': USE_REFERENCE_ELECTRODE,
    'ce_windows':            CE_WINDOWS,
    'stride':                STRIDE,
    'seed':                  SEED,
    'n_temporal_features':   int(X_temp_all.shape[1]) if 'X_temp_all' in dir() else None,
    'n_scalar_features':     int(X_scal_all.shape[1]) if 'X_scal_all' in dir() else None,
    'temporal_feature_names': feat_names_temporal,
    'results': {
        mname: {
            'acc':   float(r['acc']),
            'f1_pl': float(r['f1_pl']),
            'f1_st': float(r['f1_st']),
        }
        for mname, r in results.items()
    } if results else {},
    'onset_lags_seconds': {
        f'CE_w{w}': {
            'mean':   float(np.mean(onset_lags[w])) if onset_lags[w] else None,
            'median': float(np.median(onset_lags[w])) if onset_lags[w] else None,
        }
        for w in CE_WINDOWS
    },
}

with open(os.path.join(out_dir, 'temporal_analysis_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved: temporal_analysis_summary.json')

# ── Print final summary ────────────────────────────────────────────────────────
print('\n' + '='*60)
print('FINAL SUMMARY')
print('='*60)
print(f'\nFiles saved to: {os.path.abspath(out_dir)}')
print('  temporal_mlp_weights.pt          — temporal MLP model weights')
print('  combined_mlp_weights.pt          — combined model weights')
print('  ce_window_grid_search.csv        — grid search results')
print('  temporal_onset_profiles.png      — onset profile plots')
print('  temporal_analysis_summary.json   — key numbers for handover')

print('\nTo continue in a new chat or on a new dataset:')
print('  1. Edit DATA_DIR and CHANNELS in Section 1')
print('  2. Run all cells in order')
print('  3. Upload temporal_analysis_summary.json to the new chat')
print('     alongside HANDOVER_COMPLETE.md from the main pipeline')


---
## Section 11 — Cross-entropy coupling analysis

### The question this section answers

Sections 9 looked at each entropy type **in isolation** — the average shape
of CE, sigma, or phen_ent around plating onset. This section asks a
different question: **do the entropies move together in a specific order?**

CE, sigma, and phen_ent are computed from the same underlying voltage and
current signal, but they measure fundamentally different physical concepts:

| Entropy | What it measures | Physical lens |
|---|---|---|
| CE(ΔV) | Unpredictability of voltage increments | Information / signal complexity |
| sigma (σ) | Joule heating + overpotential losses | Thermodynamic dissipation (heat) |
| phen_ent | |ΔV·q| ± |Δq·V| | Empirical electrochemical energy flow |

If plating has a genuine **causal signature** — rather than the three
entropies just happening to correlate because they share inputs — we would
expect to see a **consistent lead-lag relationship**: one entropy
consistently changes a fixed number of seconds before another, every time
plating begins. That would suggest a real physical sequence (e.g. resistance
rises first → heat dissipation rises → only later does the voltage signal
become locally unpredictable).

If instead all three entropies change at the same instant with no
consistent ordering, that suggests they are responding to the same root
cause simultaneously rather than forming a causal chain — still useful for
detection, but not "linked behaviour" in the sense you're asking about.

### Method

For every plating onset event (same alignment as Section 9):
1. Extract the slope (1st derivative) of CE, sigma, and phen_ent in a window
   around onset
2. Compute the **cross-correlation** between every pair of entropy slopes
   at lags from −10 to +10 windows (−300s to +300s)
3. Record the lag at peak correlation for each pair, for each onset event
4. Test whether that lag is **consistent across events** (low variance) or
   essentially random (high variance, no real lead-lag structure)
5. Repeat the same analysis for stripping onset (anode returning above 0V
   during Rest) since the physical mechanism is different (dissolution vs
   deposition) and may show a different coupling pattern

### Why this is different from Section 9's `CE_phen_lag` feature

Section 3 already computes a single lag value per window as a feature for
the classifier. This section goes further: it treats the lag as the
**object of study**, not just a feature, and asks whether it is *stable*
across many onset events — which is the actual test of whether there is a
genuine joint mechanism rather than coincidental correlation.


In [ ]:
# =============================================================================
# SECTION 11.1 — EXTRACT ALIGNED ENTROPY TRACES AROUND ONSET EVENTS
# =============================================================================
#
# This re-uses the onset alignment logic from Section 9, but this time we
# keep ALL THREE entropy traces (CE, sigma, phen) together per event so we
# can study their JOINT behaviour rather than each one's marginal average.
#
# We do this separately for:
#   PLATING onset  — anode crosses BELOW 0V during charge
#   STRIPPING onset — anode crosses ABOVE 0V during rest (recovery begins)
#
# These are mechanistically different events (deposition begins vs
# dissolution begins) so we expect potentially different coupling patterns.

CROSS_WINDOW_BEFORE = 15   # windows before the transition (450s at STRIDE=30)
CROSS_WINDOW_AFTER  = 15   # windows after the transition  (450s at STRIDE=30)
MAX_LAG_WINDOWS      = 10   # max lag to test in cross-correlation (±300s)

def extract_onset_triplet(df, channel, event_type='plating'):
    """
    Extract aligned (CE, sigma, phen) traces around either plating onset
    or stripping onset events.

    Parameters
    ----------
    df         : pd.DataFrame  channel timeseries (RPT, full session)
    channel    : int           channel number (for logging only)
    event_type : str           'plating' or 'stripping'
                  'plating'   — looks in CCCV_Chg steps, onset = anode
                                first crosses BELOW 0V
                  'stripping' — looks in Rest steps, onset = anode first
                                crosses ABOVE 0V (recovery / dissolution
                                beginning to complete) — we align to the
                                START of the rest period instead, since
                                stripping is active from the first row
                                of rest if plating occurred during charge

    Returns
    -------
    events : list of dicts, each containing:
        'ce_small'  : np.ndarray  CE trace at the smallest window size
        'ce_large'  : np.ndarray  CE trace at the largest window size
        'sigma'     : np.ndarray  sigma trace
        'phen'      : np.ndarray  phen_ent trace
        'session', 'cycle' : identifying info
    """
    events = []
    small_w, large_w = min(CE_WINDOWS), max(CE_WINDOWS)

    if event_type == 'plating':
        step_filter = 'CCCV_Chg'
    else:
        step_filter = 'Rest'

    active = df[df['step_name'] == step_filter].copy()
    active['R_mOhm'] = R_inst(active['overpotential_V'].fillna(0).values,
                               active['current_mA'].values)

    for (sess, cyc, sname), grp in active.groupby(
            ['session_label', 'cycle', 'step_name'], sort=False):
        grp = grp.sort_values('abs_time').reset_index(drop=True)
        N = len(grp)

        if event_type == 'plating':
            if not grp['Li_plating'].any():
                continue
            onset_row = grp['Li_plating'].values.argmax()
        else:
            # Stripping: use the first row of Rest as the reference point,
            # since stripping is typically already active at rest onset
            # if plating occurred during the preceding charge.
            if not grp['Li_stripping'].any():
                continue
            onset_row = 0   # start of rest period

        onset_win_idx = onset_row // STRIDE
        if onset_win_idx < CROSS_WINDOW_BEFORE:
            continue

        V = grp['voltage_V'].values.astype(float)
        I = grp['current_mA'].values.astype(float)
        T = grp['temperature_C'].fillna(25.0).values.astype(float)
        R = grp['R_mOhm'].values
        Q = grp['step_capacity_mAh'].values.astype(float)
        R = np.where(np.isnan(R), np.nanmedian(R) if not np.all(np.isnan(R)) else 100, R)

        direction = 'charge' if event_type == 'plating' else 'discharge'

        ce_small = batch_ce(V, small_w)
        ce_large = batch_ce(V, large_w)
        sig      = batch_sigma(V, I, T, R, SIGMA_WIN)
        phen     = batch_phen(Q, V, T, direction, PHEN_WIN)

        s_idx = onset_win_idx - CROSS_WINDOW_BEFORE
        e_idx = onset_win_idx + CROSS_WINDOW_AFTER
        min_len = min(len(ce_small), len(ce_large), len(sig), len(phen))

        if e_idx > min_len:
            continue

        def get_seg(trace):
            seg = trace[s_idx:e_idx]
            return pd.Series(seg).ffill().bfill().fillna(0).values

        ce_small_seg = get_seg(ce_small)
        ce_large_seg = get_seg(ce_large)
        sig_seg      = get_seg(sig)
        phen_seg     = get_seg(phen)

        expected_len = CROSS_WINDOW_BEFORE + CROSS_WINDOW_AFTER
        if len(ce_small_seg) != expected_len:
            continue

        events.append({
            'ce_small': ce_small_seg,
            'ce_large': ce_large_seg,
            'sigma':    sig_seg,
            'phen':     phen_seg,
            'session':  sess,
            'cycle':    cyc,
        })

    return events


print('Extracting plating onset events...')
plating_events = []
for ch in [7, 8]:
    fp = os.path.join(DATA_DIR, FILE_PATTERN.format(ch=ch))
    if not os.path.exists(fp): continue
    df = pd.read_csv(fp, usecols=EXTENDED_COLS)
    df = df[df['session_type'] == 'RPT'].copy()
    evs = extract_onset_triplet(df, ch, event_type='plating')
    plating_events.extend(evs)
    print(f'  Ch{ch}: {len(evs)} plating onset events')

print('\nExtracting stripping onset events...')
stripping_events = []
for ch in [7, 8]:
    fp = os.path.join(DATA_DIR, FILE_PATTERN.format(ch=ch))
    if not os.path.exists(fp): continue
    df = pd.read_csv(fp, usecols=EXTENDED_COLS)
    df = df[df['session_type'] == 'RPT'].copy()
    evs = extract_onset_triplet(df, ch, event_type='stripping')
    stripping_events.extend(evs)
    print(f'  Ch{ch}: {len(evs)} stripping onset events')

print(f'\nTotal: {len(plating_events)} plating events, '
      f'{len(stripping_events)} stripping events')


In [ ]:
# =============================================================================
# SECTION 11.2 — PAIRWISE CROSS-CORRELATION LAG PER EVENT
# =============================================================================
#
# For every event, and every PAIR of entropy signals, we compute the lag
# (in windows) at which the cross-correlation between their SLOPES peaks.
#
# WHY SLOPES, NOT RAW VALUES?
# ─────────────────────────────
# Raw entropy traces share a common upward or downward drift around onset,
# which would dominate the cross-correlation and hide the actual timing
# relationship. The slope (1st derivative) isolates the moment of fastest
# change, which is what we care about for lead-lag timing.
#
# WHAT THE LAG MEANS
# ─────────────────────
# lag > 0 : the SECOND signal in the pair changes AFTER the first
#           (first signal leads, second signal lags)
# lag < 0 : the SECOND signal changes BEFORE the first
#           (second signal leads, first signal lags)
# lag = 0 : they change simultaneously (no detectable ordering)

def cross_corr_lag(slope_a, slope_b, max_lag):
    """
    Find the lag (in samples) at which cross-correlation between
    slope_a and slope_b is maximised.

    Returns (best_lag, peak_correlation).
    Positive lag means slope_b changes AFTER slope_a (slope_a leads).
    """
    n = len(slope_a)
    corrs = []
    lags  = list(range(-max_lag, max_lag + 1))

    for lag in lags:
        if lag >= 0:
            a_seg = slope_a[:n-lag] if lag > 0 else slope_a
            b_seg = slope_b[lag:]
        else:
            a_seg = slope_a[-lag:]
            b_seg = slope_b[:n+lag]

        if len(a_seg) < 5 or np.std(a_seg) < 1e-12 or np.std(b_seg) < 1e-12:
            corrs.append(0.0)
            continue

        c = np.corrcoef(a_seg, b_seg)[0, 1]
        corrs.append(0.0 if np.isnan(c) else c)

    corrs = np.array(corrs)
    best_idx = np.argmax(np.abs(corrs))
    return lags[best_idx], corrs[best_idx]


def analyse_pairwise_lags(events, label=''):
    """
    Compute pairwise lags for all entropy pairs across all events.

    Returns a DataFrame with one row per event, columns for each pair's
    lag and peak correlation.
    """
    pairs = [
        ('ce_small', 'ce_large'),
        ('ce_small', 'sigma'),
        ('ce_small', 'phen'),
        ('ce_large', 'sigma'),
        ('ce_large', 'phen'),
        ('sigma',    'phen'),
    ]

    rows = []
    for ev in events:
        row = {'session': ev['session'], 'cycle': ev['cycle']}
        for sig_a, sig_b in pairs:
            slope_a = np.gradient(ev[sig_a])
            slope_b = np.gradient(ev[sig_b])
            lag, corr = cross_corr_lag(slope_a, slope_b, MAX_LAG_WINDOWS)
            row[f'{sig_a}_{sig_b}_lag']  = lag * STRIDE   # convert to seconds
            row[f'{sig_a}_{sig_b}_corr'] = corr
        rows.append(row)

    df_lags = pd.DataFrame(rows)
    print(f'\n=== {label}: pairwise lag statistics (seconds) ===')
    print(f'{"Pair":28s}  {"Mean lag":>9}  {"Std lag":>8}  {"Mean |corr|":>11}  {"Consistency":>12}')
    for sig_a, sig_b in pairs:
        lag_col  = f'{sig_a}_{sig_b}_lag'
        corr_col = f'{sig_a}_{sig_b}_corr'
        lags  = df_lags[lag_col].values
        corrs = np.abs(df_lags[corr_col].values)

        mean_lag = np.mean(lags)
        std_lag  = np.std(lags)
        mean_corr = np.mean(corrs)

        # Consistency score: low std relative to the lag range tested
        # means most events agree on the same lag → real coupling.
        # High std means lags scatter randomly → coincidental correlation.
        max_possible_std = MAX_LAG_WINDOWS * STRIDE / np.sqrt(3)  # uniform dist std
        consistency = 1 - min(1.0, std_lag / max_possible_std)

        verdict = ('STRONG' if consistency > 0.6 and mean_corr > 0.3 else
                  'WEAK'   if consistency > 0.3 else 'NONE')

        print(f'  {sig_a:>10s} → {sig_b:<10s}    '
              f'{mean_lag:8.0f}s  {std_lag:7.0f}s  {mean_corr:10.3f}  {verdict:>12}')

    return df_lags


lags_plating = analyse_pairwise_lags(plating_events, 'PLATING onset')
lags_stripping = analyse_pairwise_lags(stripping_events, 'STRIPPING onset')


In [ ]:
# =============================================================================
# SECTION 11.3 — VISUALISE THE LAG DISTRIBUTIONS
# =============================================================================
#
# A tight, narrow distribution centred away from zero = strong evidence of
# a consistent lead-lag relationship (real coupling).
# A wide, flat distribution = no consistent ordering (coincidental
# correlation, or both respond to a common cause simultaneously).

pairs = [
    ('ce_small', 'ce_large', 'CE(small) → CE(large)'),
    ('ce_small', 'sigma',    'CE(small) → σ'),
    ('ce_small', 'phen',     'CE(small) → phen_ent'),
    ('ce_large', 'sigma',    'CE(large) → σ'),
    ('ce_large', 'phen',     'CE(large) → phen_ent'),
    ('sigma',    'phen',     'σ → phen_ent'),
]

fig, axes = plt.subplots(2, 6, figsize=(24, 8), sharex=True)
fig.suptitle(
    'Pairwise entropy lag distributions — plating (top) vs stripping (bottom)\n'
    'Narrow peak away from 0 = consistent lead-lag coupling.  '
    'Wide/flat = no consistent ordering.',
    fontweight='bold', fontsize=12
)

for col, (sig_a, sig_b, title) in enumerate(pairs):
    for row, (df_lags, event_label, color) in enumerate([
        (lags_plating,   'Plating',   '#D32F2F'),
        (lags_stripping, 'Stripping', '#1565C0'),
    ]):
        ax = axes[row, col]
        lag_col = f'{sig_a}_{sig_b}_lag'
        if lag_col not in df_lags.columns or df_lags.empty:
            ax.set_title(f'{title}\n(no data)', fontsize=8)
            continue

        vals = df_lags[lag_col].values
        ax.hist(vals, bins=np.arange(-MAX_LAG_WINDOWS*STRIDE - STRIDE/2,
                                       MAX_LAG_WINDOWS*STRIDE + STRIDE,
                                       STRIDE),
                color=color, alpha=0.7, edgecolor='white')
        ax.axvline(0, color='black', ls='--', lw=1, alpha=0.6)
        ax.axvline(np.mean(vals), color='gold', lw=2, alpha=0.9,
                  label=f'mean={np.mean(vals):.0f}s')

        if row == 0:
            ax.set_title(title, fontsize=9, fontweight='bold')
        if col == 0:
            ax.set_ylabel(f'{event_label}\nCount', fontsize=9)
        ax.legend(fontsize=6.5)
        ax.tick_params(labelsize=7)

axes[1, 2].set_xlabel('Lag (seconds)  —  positive = second signal lags first', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'cross_entropy_lag_distributions.png'),
            dpi=140, bbox_inches='tight')
plt.show()
print('Saved: cross_entropy_lag_distributions.png')


In [ ]:
# =============================================================================
# SECTION 11.4 — JOINT TRAJECTORY PLOT (phase-space view)
# =============================================================================
#
# Rather than looking at each entropy over time separately, this plots
# pairs of entropy SLOPES against each other as a trajectory through time.
# This is a "phase portrait" — a classic way to visualise coupled systems.
#
# If two entropies are genuinely coupled (one drives the other), the
# trajectory through onset will trace a consistent, repeatable LOOP or ARC
# across many events — not a random scatter.
#
# We focus on the pair with the strongest consistency score from Section 11.2.

# Identify the most consistent pair automatically
pair_scores = []
for sig_a, sig_b, title in pairs:
    lag_col  = f'{sig_a}_{sig_b}_lag'
    corr_col = f'{sig_a}_{sig_b}_corr'
    if lag_col in lags_plating.columns:
        std_lag   = lags_plating[lag_col].std()
        mean_corr = np.abs(lags_plating[corr_col]).mean()
        pair_scores.append((sig_a, sig_b, title, mean_corr, std_lag))

pair_scores.sort(key=lambda x: (-x[3], x[4]))   # high corr, low std first
best_pair = pair_scores[0]
print(f'Most consistent pair (plating): {best_pair[2]}  '
      f'(mean |corr|={best_pair[3]:.3f}, lag std={best_pair[4]:.0f}s)')

sig_a, sig_b, title = best_pair[0], best_pair[1], best_pair[2]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f'Joint trajectory: {title}\n'
             'Each line = one onset event  |  Star = onset moment  |  '
             'Dot = trajectory start',
             fontweight='bold', fontsize=11)

for ax, events, event_label, cmap_name in [
    (axes[0], plating_events,   'Plating',   'autumn'),
    (axes[1], stripping_events, 'Stripping', 'winter'),
]:
    cmap = plt.get_cmap(cmap_name)
    n_show = min(15, len(events))   # show up to 15 individual trajectories

    for i, ev in enumerate(events[:n_show]):
        slope_a = np.gradient(ev[sig_a])
        slope_b = np.gradient(ev[sig_b])

        # Normalise each event's slopes to comparable scale
        sa = (slope_a - slope_a.mean()) / (slope_a.std() + 1e-9)
        sb = (slope_b - slope_b.mean()) / (slope_b.std() + 1e-9)

        color = cmap(i / max(1, n_show - 1))
        ax.plot(sa, sb, color=color, alpha=0.5, lw=1.2)
        ax.scatter(sa[0], sb[0], color=color, s=20, marker='o', zorder=3)
        ax.scatter(sa[CROSS_WINDOW_BEFORE], sb[CROSS_WINDOW_BEFORE],
                  color=color, s=80, marker='*', zorder=4,
                  edgecolor='black', linewidth=0.5)

    ax.set_xlabel(f'{sig_a} slope (normalised)')
    ax.set_ylabel(f'{sig_b} slope (normalised)')
    ax.set_title(f'{event_label}  (n={n_show} events shown)', fontsize=9)
    ax.axhline(0, color='grey', lw=0.5, alpha=0.5)
    ax.axvline(0, color='grey', lw=0.5, alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'joint_entropy_trajectory.png'),
            dpi=140, bbox_inches='tight')
plt.show()
print('Saved: joint_entropy_trajectory.png')
print()
print('How to read this plot:')
print('  If the coloured lines from different events trace SIMILAR paths')
print('  (similar shape, similar quadrant sequence), that is strong visual')
print('  evidence of a repeatable joint mechanism.')
print('  If the lines go in random, unrelated directions, the two entropies')
print('  are not meaningfully coupled despite any marginal correlation.')


In [ ]:
# =============================================================================
# SECTION 11.5 — SUMMARY VERDICT AND FEATURE RECOMMENDATION
# =============================================================================
#
# This cell synthesises the lag consistency analysis into a plain verdict:
# is there genuine tandem behaviour between any pair of entropies, and if
# so, should it be added as a feature to the classifier?

print('='*70)
print('CROSS-ENTROPY COUPLING — SUMMARY VERDICT')
print('='*70)

def summarise_pair(df_lags, sig_a, sig_b, title, event_name):
    lag_col  = f'{sig_a}_{sig_b}_lag'
    corr_col = f'{sig_a}_{sig_b}_corr'
    if lag_col not in df_lags.columns or df_lags.empty:
        return None
    lags  = df_lags[lag_col].values
    corrs = np.abs(df_lags[corr_col].values)
    mean_lag, std_lag = np.mean(lags), np.std(lags)
    mean_corr = np.mean(corrs)
    max_possible_std = MAX_LAG_WINDOWS * STRIDE / np.sqrt(3)
    consistency = 1 - min(1.0, std_lag / max_possible_std)
    return {
        'event': event_name, 'pair': title,
        'mean_lag_s': mean_lag, 'std_lag_s': std_lag,
        'mean_corr': mean_corr, 'consistency': consistency,
    }

summary_rows = []
for sig_a, sig_b, title in pairs:
    r1 = summarise_pair(lags_plating,   sig_a, sig_b, title, 'Plating')
    r2 = summarise_pair(lags_stripping, sig_a, sig_b, title, 'Stripping')
    if r1: summary_rows.append(r1)
    if r2: summary_rows.append(r2)

summary_df = pd.DataFrame(summary_rows).sort_values(
    ['consistency', 'mean_corr'], ascending=False)

print()
print(summary_df.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

print()
strong = summary_df[(summary_df['consistency'] > 0.6) & (summary_df['mean_corr'] > 0.3)]
if not strong.empty:
    print('STRONG TANDEM BEHAVIOUR FOUND:')
    for _, row in strong.iterrows():
        direction = 'leads' if row['mean_lag_s'] > 0 else 'lags'
        print(f'  [{row["event"]}]  {row["pair"]}:  '
              f'mean lag={row["mean_lag_s"]:.0f}s  '
              f'consistency={row["consistency"]:.2f}  '
              f'|corr|={row["mean_corr"]:.2f}')
    print()
    print('RECOMMENDATION: add the lag of the strongest pair as a new feature')
    print('  e.g. "ce_small_phen_lag_consistency" — for each window, compute')
    print('  the local cross-correlation lag over the preceding N windows.')
    print('  This converts a population-level finding into a per-window')
    print('  feature usable by the classifier.')
else:
    print('NO STRONG, CONSISTENT TANDEM BEHAVIOUR FOUND across events.')
    print('The entropies likely respond to the same underlying cause')
    print('(anode potential crossing 0V) at approximately the same time,')
    print('rather than forming a detectable causal chain.')
    print()
    print('This does NOT mean the entropies are individually useless —')
    print('it means their COMBINATION should be used as simultaneous')
    print('evidence (as the existing classifier already does) rather than')
    print('as a sequential pattern feature.')

summary_df.to_csv(os.path.join(DATA_DIR, 'cross_entropy_coupling_summary.csv'),
                  index=False)
print(f'\nSaved: cross_entropy_coupling_summary.csv')
